In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import os

# ==========================================
# 1. 데이터 준비 (엑셀 파일 경로 설정)
# ==========================================
# 분석할 엑셀 파일의 경로를 지정해주세요.
file_path = '/content/연도별통계(1975-2024) (2).xlsx'

def load_and_train_model_from_excel(path):
    """엑셀 파일을 로드하고 선형 회귀 모델을 학습시키는 함수"""
    try:
        # 엑셀 파일 로드
        # sheet_name=None을 주면 모든 시트를 딕셔너리 형태로 가져옵니다.
        # 특정 시트 이름을 안다면 sheet_name='방한 외래관광객' 처럼 지정하는 것이 정확합니다.
        xls = pd.read_excel(path, sheet_name=None, engine='openpyxl')

        target_df = None

        # '방한 외래관광객'이나 'Inbound' 키워드가 있는 시트를 찾거나,
        # 시트가 하나라면 그 시트를 사용합니다.
        for sheet_name, df in xls.items():
            if '방한' in sheet_name or '외래' in sheet_name or 'Inbound' in sheet_name:
                target_df = df
                print(f"'{sheet_name}' 시트 데이터를 사용합니다.")
                break

        # 자동으로 못 찾은 경우 첫 번째 시트 사용
        if target_df is None:
            first_sheet = list(xls.keys())[0]
            target_df = xls[first_sheet]
            print(f"시트를 특정하지 못해 첫 번째 시트('{first_sheet}')를 사용합니다.")

        # -------------------------------------------------------
        # 데이터 전처리 (헤더 찾기 및 정리)
        # -------------------------------------------------------
        header_row = 0
        # 데이터프레임의 앞부분을 훑어서 'Year'와 '계(Total)'가 있는 행을 헤더로 지정
        for i, row in target_df.head(15).iterrows():
            row_str = row.astype(str).values
            if (any('Year' in s for s in row_str) or any('연' in s for s in row_str)) and \
               (any('Total' in s for s in row_str) or any('계' in s for s in row_str)):
                header_row = i + 1 # 엑셀은 인덱스가 0부터 시작하지만 read_excel에서 header 지정시 편의를 위해 조정
                break

        # 헤더를 다시 지정하여 읽기 (해당 시트만 다시 로드)
        # 위에서 찾은 시트 이름을 사용
        sheet_to_load = [k for k, v in xls.items() if v.equals(target_df)][0]
        df = pd.read_excel(path, sheet_name=sheet_to_load, header=header_row, engine='openpyxl')

        # 컬럼명 정리
        df.columns = [str(c).replace('\n', '').strip() for c in df.columns]

        # 'Year', 'Total' 컬럼 찾기
        year_col = next((c for c in df.columns if 'Year' in c or '연' in c), None)
        total_col = next((c for c in df.columns if 'Total' in c or '계' in c), None)

        if not year_col or not total_col:
            print("필요한 컬럼(연도, 계)을 찾을 수 없습니다.")
            return None

        df = df[[year_col, total_col]].copy()
        df.columns = ['Year', 'Total']

        # 숫자 변환 및 결측치 제거
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        df['Total'] = df['Total'].astype(str).str.replace(',', '').replace('-', '')
        df['Total'] = pd.to_numeric(df['Total'], errors='coerce')
        df = df.dropna()

        # -------------------------------------------------------
        # 모델 학습
        # -------------------------------------------------------
        X = df[['Year']]
        y = df['Total']

        model = LinearRegression()
        model.fit(X, y)

        return model

    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {path}")
        return None
    except Exception as e:
        print(f"오류 발생: {e}")
        return None

# 모델 학습 실행
model = load_and_train_model_from_excel(file_path)

if model:
    print("="*40)
    print("방한 외래관광객 수 예측 프로그램 (엑셀 버전)")
    print("="*40)

    while True:
        try:
            input_str = input("예측하고 싶은 연도를 입력하세요 (예: 2025, 종료하려면 'q'): ")

            if input_str.lower() == 'q':
                print("프로그램을 종료합니다.")
                break

            target_year = int(input_str)

            if target_year < 2025:
                print("2025년 이후의 연도를 입력해주세요.")
                continue

            prediction = model.predict([[target_year]])[0]

            print(f"▶ {target_year}년 예상 방한 외래관광객 수: 약 {int(prediction):,} 명")
            print("-" * 40)

        except ValueError:
            print("올바른 숫자를 입력해주세요.")
else:
    print("모델 생성 실패. 파일 경로와 내용을 확인해주세요.")

'방한 외래관광객' 시트 데이터를 사용합니다.
방한 외래관광객 수 예측 프로그램 (엑셀 버전)
예측하고 싶은 연도를 입력하세요 (예: 2025, 종료하려면 'q'): 2030


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


▶ 2030년 예상 방한 외래관광객 수: 약 13,564,549 명
----------------------------------------
예측하고 싶은 연도를 입력하세요 (예: 2025, 종료하려면 'q'): q
프로그램을 종료합니다.
